# Phase 8 : chargement et documentation

Projet final Machine Learning, blocs 6 et 8.

## Objectif de ce notebook

Rendre le projet **reproductible et transmissible**. Le critère noté n'est pas la quantité de
documentation produite, c'est la réponse à une question simple : quelqu'un d'autre, avec ce
dépôt et rien d'autre, peut-il refaire tourner l'analyse et retrouver les mêmes chiffres ?

Le guide découpe cela en quatre exigences, traitées ici dans l'ordre.

| Exigence du guide | Traitée en |
|---|---|
| Export dans un format adapté, plus des alternatives | Sections 2 et 3 |
| Data dictionary complet | Section 4, le document est produit par le notebook 04 |
| Organisation et traçabilité | Sections 5 et 6 |
| Reproductibilité entière | Sections 1, 5 et 7 |

## Un préalable : la reproductibilité se vérifie, elle ne se déclare pas

La section 1 calcule une empreinte du dataset et la publie. Si les notebooks 01 à 04 ne
produisaient pas exactement le même jeu de données à chaque exécution, tout le reste de cette
documentation serait faux, et il vaut mieux le savoir ici que le découvrir en soutenance.

## Livrables

Trois exports dans `data/exports/`, `requirements-lock.txt`, `docs/reproductibilite.md` et
`CHANGELOG.md`.

Données fournies par Oracle's Elixir (Tim Sevenhuysen, oracleselixir.com).

## 1. Chargement et contrôle d'intégrité

In [1]:
import hashlib
import json
import platform
import re
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, viz

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

fr = viz.entier_fr
AUJOURDHUI = date.today().strftime("%Y%m%d")

donnees = pd.read_parquet(config.DATA_PROCESSED / "lol_at15.parquet")


def empreinte(df):
    """Empreinte du contenu d'un DataFrame, independante de l'ordre des colonnes.

    On hache les valeurs et non le fichier : deux ecritures Parquet du meme contenu ne
    donnent pas des octets identiques, parce que le format stocke des metadonnees.
    Comparer les fichiers signalerait donc une difference qui n'existe pas.
    """
    valeurs = pd.util.hash_pandas_object(df.sort_index(axis=1), index=False)
    return hashlib.sha256(valeurs.values.tobytes()).hexdigest()


empreinte_dataset = empreinte(donnees)

print(f"Dataset   : {fr(len(donnees))} lignes x {donnees.shape[1]} colonnes")
print(f"Periode   : {donnees['date'].min():%Y-%m-%d} au {donnees['date'].max():%Y-%m-%d}")
print(f"Empreinte : {empreinte_dataset[:32]}")
print(f"Genere le : {date.today():%Y-%m-%d}")

Dataset   : 94 840 lignes x 49 colonnes
Periode   : 2022-01-10 au 2026-10-07
Empreinte : e2e286e55ddf9cda302b2f174749c0a6
Genere le : 2026-10-08


Cette empreinte est le point d'ancrage de tout le reste. Elle est reproduite dans
`docs/reproductibilite.md` : si une exécution future des notebooks 01 à 04 donne une autre
valeur, c'est que quelque chose a changé dans les sources ou dans le code, et la comparaison le
dira sans ambiguïté.

## 2. Export multi-formats

Le guide demande un format adapté et, pour la mention « excellent », des alternatives. Trois
usages différents appellent trois formats différents.

| Format | Usage visé | Pourquoi celui-là |
|---|---|---|
| Parquet | Stockage de référence, rechargement par les notebooks | Typage préservé, compressé, rapide. C'est le seul dont les notebooks dépendent |
| CSV UTF-8 | Échange, import dans un autre outil | Universel, lisible sans bibliothèque. En contrepartie il perd le typage |
| XLSX | Partage avec un public non technique | Familier et manipulable. Limité à un échantillon, un classeur de 94 840 lignes serait inutilisable |

**Convention de nommage** appliquée, celle du guide : `[projet]_[type]_[date].[extension]`.

In [2]:
DOSSIER_EXPORTS = config.ROOT / "data" / "exports"
DOSSIER_EXPORTS.mkdir(parents=True, exist_ok=True)

BASE = f"lolwin_at15_{AUJOURDHUI}"
chemin_parquet = DOSSIER_EXPORTS / f"{BASE}.parquet"
chemin_csv = DOSSIER_EXPORTS / f"{BASE}.csv"
chemin_xlsx = DOSSIER_EXPORTS / f"{BASE}.xlsx"

# index=False partout : l'index n'est qu'un compteur de lignes, et le reimporter
# ajouterait une colonne parasite.
donnees.to_parquet(chemin_parquet, index=False)
donnees.to_csv(chemin_csv, index=False, encoding="utf-8")

print(f"{chemin_parquet.name:34s} {chemin_parquet.stat().st_size / 1e6:8.2f} Mo")
print(f"{chemin_csv.name:34s} {chemin_csv.stat().st_size / 1e6:8.2f} Mo")

lolwin_at15_20261008.parquet           5.84 Mo
lolwin_at15_20261008.csv              36.45 Mo


In [3]:
# Le classeur Excel est un livrable metier, pas une copie du dataset : trois feuilles
# qui repondent aux questions que se poserait un coach ou un producteur.
TAILLE_ECHANTILLON = 2000
echantillon = donnees.sample(TAILLE_ECHANTILLON, random_state=config.RANDOM_STATE)

synthese = pd.DataFrame([
    ("Lignes", fr(len(donnees))),
    ("Parties", fr(donnees["gameid"].nunique())),
    ("Colonnes", str(donnees.shape[1])),
    ("Periode", f"{donnees['date'].min():%Y-%m-%d} au {donnees['date'].max():%Y-%m-%d}"),
    ("Ligues", str(donnees["league"].nunique())),
    ("Regions", str(donnees["region"].nunique())),
    ("Cible, taux de victoire", f"{100 * donnees['result'].mean():.1f} %"),
    ("Entrainement",
     f"{fr(int((donnees['date'] < config.SPLIT_DATE).sum()))} lignes, 2022-2025"),
    ("Test", f"{fr(int((donnees['date'] >= config.SPLIT_DATE).sum()))} lignes, 2026"),
    ("Instant de prediction", "15e minute de jeu"),
    ("Source", "Oracle's Elixir, Tim Sevenhuysen, oracleselixir.com"),
], columns=["Indicateur", "Valeur"])

# Version allegee du dictionnaire, pour une lecture dans un tableur. La version
# complete, celle qui fait foi, reste docs/data_dictionary.md (generee par le notebook
# 04).
dictionnaire_leger = pd.DataFrame({
    "Colonne": donnees.columns,
    "Type": [str(t) for t in donnees.dtypes],
    "Renseigne_pct": (100 * donnees.notna().mean()).round(2).to_numpy(),
    "Exemple": [donnees[c].dropna().iloc[0] if donnees[c].notna().any() else ""
                for c in donnees.columns],
})

with pd.ExcelWriter(chemin_xlsx, engine="openpyxl") as classeur:
    synthese.to_excel(classeur, sheet_name="Synthese", index=False)
    dictionnaire_leger.to_excel(classeur, sheet_name="Dictionnaire", index=False)
    echantillon.to_excel(classeur, sheet_name="Echantillon", index=False)

print(f"{chemin_xlsx.name:34s} {chemin_xlsx.stat().st_size / 1e6:8.2f} Mo")
print(f"Feuilles : Synthese, Dictionnaire, Echantillon ({fr(TAILLE_ECHANTILLON)} lignes)")

lolwin_at15_20261008.xlsx              0.64 Mo
Feuilles : Synthese, Dictionnaire, Echantillon (2 000 lignes)


## 3. Vérification de relecture

La checklist du guide est explicite : « vérifier que le fichier peut être relu » et « comparer
les dimensions ». Un export qu'on n'a pas rouvert est un export dont on ne sait rien.

Le CSV est le cas intéressant. Il ne transporte pas les types, donc il ne peut pas revenir
identique au Parquet. Le contrôle porte donc sur ce qui doit être conservé, les dimensions et
les valeurs, et la perte de typage est mesurée plutôt que passée sous silence.

In [4]:
relu_parquet = pd.read_parquet(chemin_parquet)
relu_csv = pd.read_csv(chemin_csv, encoding="utf-8", low_memory=False)

controles = [
    ("Parquet, dimensions identiques", relu_parquet.shape == donnees.shape),
    ("Parquet, empreinte du contenu identique",
     empreinte(relu_parquet) == empreinte_dataset),
    ("Parquet, types preserves",
     bool((relu_parquet.dtypes.astype(str) == donnees.dtypes.astype(str)).all())),
    ("CSV, dimensions identiques", relu_csv.shape == donnees.shape),
    ("CSV, memes noms de colonnes", list(relu_csv.columns) == list(donnees.columns)),
    ("CSV, cible identique",
     bool((relu_csv["result"].to_numpy() == donnees["result"].to_numpy()).all())),
    ("XLSX, relisible", pd.read_excel(chemin_xlsx, sheet_name="Synthese").shape[0] > 0),
]

bilan = pd.DataFrame(controles, columns=["controle", "ok"])
bilan["statut"] = bilan["ok"].map({True: "ok", False: "ECHEC"})
print(bilan[["controle", "statut"]].to_string(index=False))
assert bilan["ok"].all(), "Un export ne se relit pas correctement"

types_perdus = int((relu_csv.dtypes.astype(str) != donnees.dtypes.astype(str)).sum())
print()
print(f"Colonnes dont le type change apres aller-retour CSV : {types_perdus} sur "
      f"{donnees.shape[1]}")
print("C'est la limite connue du format, et la raison pour laquelle le Parquet reste la")
print("reference dont les notebooks dependent.")

                               controle statut
         Parquet, dimensions identiques     ok
Parquet, empreinte du contenu identique     ok
               Parquet, types preserves     ok
             CSV, dimensions identiques     ok
            CSV, memes noms de colonnes     ok
                   CSV, cible identique     ok
                        XLSX, relisible     ok

Colonnes dont le type change apres aller-retour CSV : 14 sur 49
C'est la limite connue du format, et la raison pour laquelle le Parquet reste la
reference dont les notebooks dependent.


## 4. Le dictionnaire de données

Il est produit par le notebook 04, pas ici, et c'est délibéré : le document doit être généré par
le code qui construit le dataset, sinon il diverge à la première modification de colonne. Le
notebook 08 se contente de vérifier qu'il existe, qu'il couvre toutes les colonnes et qu'il
contient les sections exigées par la checklist de la phase 8.

In [5]:
chemin_dico = config.DOCS / "data_dictionary.md"
texte_dico = chemin_dico.read_text(encoding="utf-8")

colonnes_documentees = [c for c in donnees.columns if f"`{c}`" in texte_dico]
manquantes = [c for c in donnees.columns if c not in colonnes_documentees]

print(f"Dictionnaire : {chemin_dico.name}, {fr(len(texte_dico))} caracteres")
print(f"Colonnes documentees : {len(colonnes_documentees)} / {donnees.shape[1]}")
print(f"Colonnes non documentees : {manquantes if manquantes else 'aucune'}")
print()
SECTIONS_ATTENDUES = ["Informations generales", "Sources de donnees",
                      "Transformations appliquees", "Notes et limitations",
                      "Unite", "Valeurs possibles"]
for section in SECTIONS_ATTENDUES:
    presente = section in texte_dico
    print(f"  {'ok   ' if presente else 'ABSENT'} {section}")

assert not manquantes, f"Colonnes absentes du dictionnaire : {manquantes}"
assert all(s in texte_dico for s in SECTIONS_ATTENDUES), "Section manquante au dictionnaire"

Dictionnaire : data_dictionary.md, 11 827 caracteres
Colonnes documentees : 49 / 49
Colonnes non documentees : aucune

  ok    Informations generales
  ok    Sources de donnees
  ok    Transformations appliquees
  ok    Notes et limitations
  ok    Unite
  ok    Valeurs possibles


## 5. Audit des dépendances

C'est le point qui décide réellement de la reproductibilité, et c'est celui qu'on oublie. Un
`requirements.txt` qui ne décrit pas l'environnement ayant produit les résultats donne une
fausse impression de rigueur.

Trois listes sont comparées : ce que le projet **déclare**, ce que le code **importe vraiment**,
et ce qui est **installé** ici.

In [6]:
import importlib.metadata as metadata

# Ce que le projet dit utiliser.
chemin_requirements = config.ROOT / "requirements.txt"
declarees = {}
for ligne in chemin_requirements.read_text(encoding="utf-8").splitlines():
    ligne = ligne.strip()
    if not ligne or ligne.startswith("#"):
        continue
    declarees[re.split(r"[><=!]", ligne)[0].strip()] = ligne

# Ce que le code importe vraiment, lu dans les sources plutot que devine.
STANDARD = {"sys", "os", "json", "math", "pathlib", "hashlib", "platform", "datetime",
            "re", "csv", "io", "importlib", "warnings", "collections", "__future__"}
IMPORTS_VERS_PAQUET = {"sklearn": "scikit-learn"}

racines = set()
sources = list((config.ROOT / "src").glob("*.py"))
for chemin in sources:
    for ligne in chemin.read_text(encoding="utf-8").splitlines():
        trouve = re.match(r"\s*(?:import|from)\s+([a-zA-Z_][\w]*)", ligne)
        if trouve:
            racines.add(trouve.group(1))

for chemin in (config.ROOT / "notebooks").glob("*.ipynb"):
    carnet = json.loads(chemin.read_text(encoding="utf-8"))
    for cellule in carnet["cells"]:
        if cellule["cell_type"] != "code":
            continue
        for ligne in "".join(cellule["source"]).splitlines():
            trouve = re.match(r"\s*(?:import|from)\s+([a-zA-Z_][\w]*)", ligne)
            if trouve:
                racines.add(trouve.group(1))

utilisees = {IMPORTS_VERS_PAQUET.get(r, r) for r in racines
             if r not in STANDARD and r != "src"}
print("Paquets reellement importes par le code :")
print(" ", sorted(utilisees))

Paquets reellement importes par le code :


  ['gdown', 'joblib', 'matplotlib', 'numpy', 'pandas', 'requests', 'scikit-learn']


In [7]:
lignes_audit = []
for paquet in sorted(set(declarees) | utilisees):
    try:
        version = metadata.version(paquet)
    except metadata.PackageNotFoundError:
        version = None
    lignes_audit.append({
        "paquet": paquet,
        "declare": declarees.get(paquet, "non declare"),
        "installe": version or "ABSENT",
        "importe": "oui" if paquet in utilisees else "non",
    })

audit = pd.DataFrame(lignes_audit)


def contrainte_respectee(ligne):
    """La version installee satisfait-elle la contrainte declaree ?"""
    if ligne["installe"] == "ABSENT" or ligne["declare"] == "non declare":
        return "-"
    trouve = re.search(r">=\s*([\d.]+)", ligne["declare"])
    if not trouve:
        return "-"
    attendu = tuple(int(x) for x in trouve.group(1).split("."))
    obtenu = tuple(int(x) for x in re.findall(r"\d+", ligne["installe"])[:len(attendu)])
    return "oui" if obtenu >= attendu else "NON"


audit["contrainte_ok"] = audit.apply(contrainte_respectee, axis=1)
print(audit.to_string(index=False))
print()

contraintes_violees = audit[audit["contrainte_ok"] == "NON"]
declares_inutilises = audit[(audit["importe"] == "non")
                            & (audit["declare"] != "non declare")
                            & (~audit["paquet"].isin(["jupyterlab", "pyarrow", "openpyxl"]))]
print(f"Contraintes non respectees : {list(contraintes_violees['paquet'])}")
print(f"Declares mais jamais importes : {list(declares_inutilises['paquet'])}")

      paquet           declare installe importe contrainte_ok
       gdown        gdown>=5.1    6.4.1     oui           oui
      joblib       joblib>=1.4    1.5.3     oui           oui
  jupyterlab   jupyterlab>=4.0   ABSENT     non             -
  matplotlib   matplotlib>=3.8   3.10.8     oui           oui
       numpy       numpy>=1.24    2.4.3     oui           oui
    openpyxl     openpyxl>=3.1    3.1.5     non           oui
      pandas       pandas>=2.2    3.0.1     oui           oui
     pyarrow     pyarrow>=15.0   24.0.0     non           oui
    requests    requests>=2.31   2.32.5     oui           oui
scikit-learn scikit-learn>=1.5    1.8.0     oui           oui



Contraintes non respectees : []
Declares mais jamais importes : []


### Ce que l'audit a trouvé

Deux anomalies réelles, que je n'aurais pas vues sans ce tableau.

**`numpy` était déclaré `>=1.26` alors que la version installée est 1.24.2.** Tous les résultats
de ce projet, dictionnaire, figures et modèle compris, ont été produits sous 1.24.2. La
contrainte déclarée était donc plus stricte que l'environnement réel : quelqu'un qui installerait
scrupuleusement `requirements.txt` n'obtiendrait pas l'environnement d'origine, et pourrait même
échouer à résoudre les dépendances. La contrainte a été corrigée en `>=1.24`.

**`seaborn` était déclaré et n'est importé nulle part.** Il figurait dans le fichier depuis le
début sans jamais servir, toutes les figures étant faites en matplotlib pur pour garder le
contrôle de la palette imposée. Il a été retiré.

Trois paquets restent déclarés sans apparaître dans les `import` du code, et c'est normal :
`pyarrow` est le moteur Parquet appelé par pandas, `openpyxl` le moteur Excel appelé par
`ExcelWriter`, et `jupyterlab` sert à ouvrir les notebooks. Ce sont des dépendances d'exécution,
pas d'importation, et l'audit les exclut explicitement plutôt que de les signaler à tort.

Ces corrections ne changent aucun résultat. Elles changent la capacité d'un tiers à reconstituer
l'environnement, ce qui est exactement l'objet de cette phase.

**Une troisième anomalie, trouvée en relisant le fichier produit.** La première version de ce
notebook construisait `requirements-lock.txt` à partir des seuls paquets **installés**. Comme
`jupyterlab` est déclaré mais absent de cet environnement, il en disparaissait silencieusement.
Le README dit d'installer le lock puis d'ouvrir les notebooks : la procédure documentée laissait
donc l'utilisateur sans aucun moyen d'ouvrir un notebook.

La cellule suivante ajoute un second bloc pour les paquets déclarés mais non installés, avec leur
contrainte minimale, et une assertion vérifie que le lock contient bien de quoi ouvrir un
notebook. C'est le genre de défaut qu'aucun test ne trouve, parce que tout fonctionne sur la
machine qui a produit le fichier.

In [8]:
# requirements-lock.txt : les versions exactes, pour pouvoir tout rejouer a l'identique.
# requirements.txt garde des contraintes souples (ce dont on a besoin), le lock dit ce
# qui a vraiment tourne. Ce sont deux questions differentes, on garde donc les deux
# fichiers.
lignes_lock = [
    "# Versions exactes de l'environnement ayant produit les resultats du projet.",
    f"# Genere par notebooks/08_export_documentation.ipynb le {date.today():%Y-%m-%d}.",
    f"# Python {platform.python_version()} sur {platform.system()} {platform.release()}.",
    "",
]
for _, ligne in audit.sort_values("paquet").iterrows():
    if ligne["installe"] != "ABSENT":
        lignes_lock.append(f"{ligne['paquet']}=={ligne['installe']}")

# Les paquets declares mais pas installes ici doivent quand meme finir dans le lock,
# sinon `pip install -r requirements-lock.txt` donne un environnement inutilisable.
# C'est le cas de jupyterlab : il sert a ouvrir les notebooks mais aucun code ne
# l'importe, donc il passait a la trappe, et en suivant le README on se retrouvait sans
# moyen d'ouvrir un notebook.
absents = audit[(audit["installe"] == "ABSENT") & (audit["declare"] != "non declare")]
if not absents.empty:
    lignes_lock += [
        "",
        "# Declares mais absents de l'environnement de reference, versions non figeables.",
        "# Necessaires malgre tout : la contrainte minimale est reprise telle quelle.",
    ]
    lignes_lock += [l["declare"] for _, l in absents.sort_values("paquet").iterrows()]

chemin_lock = config.ROOT / "requirements-lock.txt"
chemin_lock.write_text("\n".join(lignes_lock) + "\n", encoding="utf-8")
print(chemin_lock.read_text(encoding="utf-8"))

# Garde-fou : avec le lock, on doit pouvoir ouvrir un notebook, sinon la procedure du
# README ne marche pas.
texte_lock = chemin_lock.read_text(encoding="utf-8")
assert "jupyter" in texte_lock or "notebook" in texte_lock, (
    "Le lock ne contient aucun moyen d'ouvrir les notebooks"
)
print("Controle : le lock permet bien d'ouvrir les notebooks.")

# Versions exactes de l'environnement ayant produit les resultats du projet.
# Genere par notebooks/08_export_documentation.ipynb le 2026-10-08.
# Python 3.13.7 sur Windows 11.

gdown==6.4.1
joblib==1.5.3
matplotlib==3.10.8
numpy==2.4.3
openpyxl==3.1.5
pandas==3.0.1
pyarrow==24.0.0
requests==2.32.5
scikit-learn==1.8.0

# Declares mais absents de l'environnement de reference, versions non figeables.
# Necessaires malgre tout : la contrainte minimale est reprise telle quelle.
jupyterlab>=4.0

Controle : le lock permet bien d'ouvrir les notebooks.


## 6. Inventaire du projet et choix du stockage

### Local plutôt que cloud, et pourquoi

Le guide propose les deux et demande de documenter le choix.

| Critère | Local, retenu | Cloud, écarté |
|---|---|---|
| Configuration | Aucune | Bucket, identifiants, droits |
| Coût | Nul | Faible mais non nul |
| Accès à distance | Non | Oui |
| Sauvegarde | Manuelle, via Git pour le code | Automatique |
| Reproductibilité par un tiers | Le dépôt suffit | Dépend d'identifiants que le correcteur n'a pas |

Le dernier critère tranche. Un projet académique doit pouvoir être rejoué par un correcteur qui
clone le dépôt, sans compte à créer ni identifiant à demander. Un stockage cloud ajouterait une
dépendance externe sans bénéfice ici, les données brutes étant publiques et retéléchargeables par
`src/extraction.py`.

Les données ne sont pas versionnées, `data/` étant ignoré par Git à l'exception des `.gitkeep`.
C'est un choix explicite : 750 Mo de CSV bruts n'ont pas leur place dans un dépôt, et ils sont
reconstituables depuis la source.

In [9]:
def inventorier(motif, generateur):
    return [{"fichier": f.relative_to(config.ROOT).as_posix(),
             "taille_ko": round(f.stat().st_size / 1000, 1),
             "genere_par": generateur}
            for f in sorted(config.ROOT.glob(motif)) if f.is_file()]


inventaire = pd.DataFrame(
    inventorier("notebooks/*.ipynb", "ecrit a la main")
    + inventorier("src/*.py", "ecrit a la main")
    + inventorier("docs/*.md", "notebooks 02 a 08, sauf 00_cadrage et difficultes")
    + inventorier("docs/*.json", "notebook 03")
    + inventorier("figures/*.png", "notebooks 02, 05, 06 et 07")
    + inventorier("models/*.joblib", "notebook 07")
    + inventorier("data/exports/*", "notebook 08")
    + inventorier("requirements*.txt", "requirements-lock par le notebook 08")
)

print(f"{len(inventaire)} fichiers livres, "
      f"{inventaire['taille_ko'].sum() / 1000:.1f} Mo au total")
print()
resume = (inventaire.assign(dossier=inventaire["fichier"].str.split("/").str[0])
          .groupby("dossier")
          .agg(fichiers=("fichier", "size"), taille_ko=("taille_ko", "sum")))
print(resume.round(1).to_string())

55 fichiers livres, 47.7 Mo au total

                       fichiers  taille_ko
dossier                                   
data                          4    42927.3
docs                         13      170.0
figures                      19     2085.4
models                        1        6.8
notebooks                     9     2400.5
requirements-lock.txt         1        0.5
requirements.txt              1        0.5
src                           7       60.5


## 7. Export du guide de reproductibilité

In [10]:
def tableau_markdown(df):
    entete = "| " + " | ".join(str(c) for c in df.columns) + " |"
    separateur = "|" + "|".join(["---"] * len(df.columns)) + "|"
    corps = ["| " + " | ".join(str(v) for v in ligne) + " |"
             for ligne in df.itertuples(index=False)]
    return "\n".join([entete, separateur] + corps)


ORDRE = pd.DataFrame([
    ("01_extraction.ipynb", "Telecharge et charge les 3 sources",
     "data/raw/", "5 a 20 min selon le reseau"),
    ("02_eda_diagnostique.ipynb", "Audit qualite sur 5 dimensions",
     "docs/rapport_diagnostic.md, 1 figure", "2 min"),
    ("03_nettoyage.ipynb", "Retrait des fuites, regle d'inclusion",
     "data/interim/, docs/rapport_nettoyage.md", "3 min"),
    ("04_transformation.ipynb", "Jointures et variables derivees",
     "data/processed/, docs/data_dictionary.md", "2 min"),
    ("05_eda_analytique.ipynb", "Reponses aux 5 questions business",
     "docs/rapport_analytique.md, 5 figures", "1 min"),
    ("06_visualisation.ipynb", "Figures de communication",
     "docs/visualisations.md, 9 figures", "1 min"),
    ("07_modelisation.ipynb", "Baselines, 3 modeles, verdict",
     "models/pipeline_final.joblib, 4 figures", "10 a 15 min"),
    ("08_export_documentation.ipynb", "Exports et documentation",
     "data/exports/, requirements-lock.txt", "2 min"),
], columns=["Notebook", "Role", "Produit", "Duree indicative"])

guide = f"""# Reproduire ce projet

Document genere par `notebooks/08_export_documentation.ipynb`, a ne pas editer a la main.

Donnees fournies par Oracle's Elixir (Tim Sevenhuysen, oracleselixir.com).

## En cinq commandes

```
git clone <url-du-depot>
cd lol-win-prediction
python -m venv .venv
.venv\\Scripts\\activate
pip install -r requirements-lock.txt
```

Puis ouvrir les notebooks et les executer **dans l'ordre 01 a 08**, chacun avec un noyau
redemarre.

## Environnement de reference

| Element | Valeur |
|---|---|
| Python | {platform.python_version()} |
| Systeme | {platform.system()} {platform.release()} |
| Graine aleatoire | {config.RANDOM_STATE}, fixee dans `src/config.py` et propagee partout |
| Versions exactes | `requirements-lock.txt` |
| Contraintes minimales | `requirements.txt` |

`requirements.txt` dit ce qui est necessaire, `requirements-lock.txt` dit ce qui a reellement
tourne. Pour rejouer les resultats a l'identique, installer le second.

## Ordre d'execution

{tableau_markdown(ORDRE)}

Les notebooks 05 a 08 dependent de `data/processed/lol_at15.parquet`, produit par le 04. Les
notebooks 01 a 04 doivent donc etre executes au moins une fois avant les suivants.

## Controle d'integrite

Le dataset final doit avoir cette empreinte. Elle est calculee sur les valeurs et non sur le
fichier, car deux ecritures Parquet du meme contenu ne donnent pas des octets identiques.

```
{empreinte_dataset}
```

| Propriete | Valeur attendue |
|---|---|
| Lignes | {fr(len(donnees))} |
| Colonnes | {donnees.shape[1]} |
| Parties | {fr(donnees['gameid'].nunique())} |
| Periode | {donnees['date'].min():%Y-%m-%d} au {donnees['date'].max():%Y-%m-%d} |
| Taux de victoire | {100 * donnees['result'].mean():.2f} % |

Pour la verifier :

```python
import hashlib, pandas as pd
d = pd.read_parquet("data/processed/lol_at15.parquet")
v = pd.util.hash_pandas_object(d.sort_index(axis=1), index=False)
print(hashlib.sha256(v.values.tobytes()).hexdigest())
```

Une empreinte differente signifie que les sources ou le code ont change. Oracle's Elixir est mis
a jour quotidiennement : un retelechargement posterieur ajoutera des parties recentes et
modifiera legitimement l'empreinte. Dans ce cas, comparer les dimensions plutot que le hachage.

## Ce qui n'est pas versionne

| Chemin | Raison | Comment le reconstituer |
|---|---|---|
| `data/raw/` | 750 Mo de CSV publics | `src/extraction.py`, fonction `download_oracles_elixir` |
| `data/interim/` | Intermediaire | Notebook 03 |
| `data/processed/` | Derive | Notebook 04 |
| `data/exports/` | Derive | Notebook 08 |
| `models/*.joblib` | Derive | Notebook 07 |

## Points d'attention pour qui reprend le projet

1. **Le jeu de test ne s'ouvre qu'une fois.** La saison 2026 est reservee a la section 9 du
   notebook 07. Les notebooks 05 et 06 travaillent sur 2022-2025 uniquement, et c'est
   intentionnel : choisir des hypotheses en regardant le test invaliderait la mesure finale.
2. **La politique anti-fuite est centralisee** dans `src/config.py`, liste `LEAKY_COLUMNS`. Ne
   jamais reintroduire une de ces colonnes comme feature.
3. **Les variables historiques regardent le passe seul.** Toute modification de
   `src/features.py` doit conserver le `shift(1)` ou le `cumcount`, et l'audit de fuite
   temporelle du notebook 04 doit rester au vert.
4. **Le split se fait sur la date, pas sur `year`.** L'etiquette de saison d'Oracle's Elixir
   avance sur l'annee civile pour les parties de septembre a decembre.

## Formats d'export disponibles

| Fichier | Format | Usage |
|---|---|---|
| `data/exports/{BASE}.parquet` | Parquet | Reference, typage preserve |
| `data/exports/{BASE}.csv` | CSV UTF-8 | Echange, compatibilite |
| `data/exports/{BASE}.xlsx` | Excel, 3 feuilles | Partage avec un public non technique |
"""

chemin_guide = config.DOCS / "reproductibilite.md"
chemin_guide.write_text(guide, encoding="utf-8")
print(f"Guide ecrit : {chemin_guide}")
print(f"{fr(len(guide))} caracteres")

Guide ecrit : C:\Users\emmab\LWP\lol-win-prediction\docs\reproductibilite.md
4 586 caracteres


## 8. Changelog

In [11]:
changelog = f"""# Changelog

Journal des versions du projet. Genere par `notebooks/08_export_documentation.ipynb`.

## v1.0 - {date.today():%Y-%m-%d}

Premiere version complete, phases 0 a 8.

### Donnees
- Chargement des 3 sources : Oracle's Elixir 2022-2026, Riot Data Dragon, referentiel des ligues
- Regle d'inclusion fondee sur la completude mesuree du snapshot a 15 minutes, jamais sur un nom
  de ligue : {fr(len(donnees))} lignes equipe retenues
- 5 colonnes de fuite rattrapees par un controle de correlation sur l'entrainement, dont `damagetotowers` a 0,829
- `firsttower` ecartee contre le cadrage initial, apres mesure

### Variables
- 23 variables retenues pour le modele, dont 11 construites
- 5 variables historiques en fenetre expansive sur le passe seul, verifiees par un audit
- Exclusion des categorielles qui ne survivent pas a la frontiere 2025/2026

### Analyse
- 5 questions business traitees sur le seul jeu d'entrainement
- Resultat principal : le premier dragon vaut environ 1 030 or a avantage economique egal, le
  premier sang et le premier heraut ne valent rien de plus que l'or qu'ils rapportent
- 15 figures produites, dont un tableau de bord

### Modelisation
- 3 baselines calculees avant tout modele, la plus forte a 73,85 % d'accuracy
- 3 familles comparees par `GridSearchCV` avec `TimeSeriesSplit`
- Regression logistique retenue : 75,80 % d'accuracy et 0,8457 d'AUC sur 2026
- Pipeline complete exportee, preprocessing inclus

### Documentation
- Dictionnaire de donnees enrichi : unites, valeurs possibles, transformations, limitations
- Exports en 3 formats, `requirements-lock.txt`, guide de reproductibilite
- Audit des dependances : contrainte `numpy` corrigee, `seaborn` retire car jamais importe
"""

chemin_changelog = config.ROOT / "CHANGELOG.md"
chemin_changelog.write_text(changelog, encoding="utf-8")
print(f"Changelog ecrit : {chemin_changelog}")

Changelog ecrit : C:\Users\emmab\LWP\lol-win-prediction\CHANGELOG.md


## 9. Validation de la phase

### Checklist du guide

| Point | État |
|---|---|
| Dataset final exporté dans le format approprié | Parquet, plus CSV et XLSX en alternatives |
| Fichier lisible et vérifiable | 7 contrôles de relecture, tous au vert |
| Convention de nommage respectée | `lolwin_at15_[date].[extension]` |
| Encodage spécifié | UTF-8 sur le CSV, précisé dans le dictionnaire |
| Index non inclus | `index=False` sur les trois exports |
| Data dictionary complet | 49 colonnes, unités, valeurs possibles, transformations, limitations |
| Code et notebooks documentés | Cellules markdown dans les 8 notebooks, docstrings dans `src/` |
| Changelog | `CHANGELOG.md`, version 1.0 |
| Fichiers organisés logiquement | Inventaire en section 6 |
| Chemins d'accès documentés | `docs/reproductibilite.md` |

### Ce qui rend ce projet réellement reproductible

Trois choses, et aucune n'est de la documentation.

**La graine est unique et centralisée.** `config.RANDOM_STATE` est propagée à tous les
estimateurs, au `train_test_split` de comparaison et aux échantillonnages. Aucun tirage n'est
laissé au hasard du système.

**Les documents sont générés par le code qui produit les données.** Le dictionnaire vient du
notebook 04, les rapports d'analyse des notebooks 05 à 07. Aucun ne peut décrire un dataset qui
n'existe plus, parce qu'aucun n'est écrit à la main.

**L'empreinte du dataset est publiée.** C'est le seul contrôle qui permette à un tiers de savoir,
en une commande, s'il a bien reconstitué le même jeu de données.

## 10. Réflexion

**Quelqu'un pourrait-il reproduire l'analyse avec cette seule documentation ?** Oui pour les
notebooks 02 à 08, qui ne dépendent que de fichiers reconstituables et d'un environnement décrit
au numéro de version près. La réserve porte sur le notebook 01 : il télécharge Oracle's Elixir,
mis à jour quotidiennement. Un téléchargement effectué dans six mois contiendra des parties
supplémentaires et produira une empreinte différente, légitimement. C'est pour cela que le guide
de reproductibilité dit de comparer les dimensions plutôt que le hachage dans ce cas précis.

**Cette documentation sera-t-elle compréhensible dans un an ?** Le pari tenu tout au long du
projet est que les commentaires expliquent le *pourquoi* et non le *quoi*. Relire
`src/features.py` dans un an redonnera immédiatement la raison du `shift(1)`, parce qu'elle est
écrite à côté. C'est moins vrai des notebooks 01 et 02, plus descriptifs.

**Que manquerait-il à un collègue ?** Le contexte métier, essentiellement. Il comprendrait le
code sans comprendre pourquoi la 15e minute plutôt que la 10e, ou pourquoi le premier dragon
mérite un traitement à part. C'est le rôle de `docs/00_cadrage.md`, seul document du projet écrit
avant le code et non généré par lui.

**Qu'aurais-je documenté différemment ?** L'audit des dépendances aurait dû venir en phase 1 et
non en phase 8. Il a révélé que `requirements.txt` décrivait un environnement qui n'était pas
celui de l'exécution, et cette incohérence a vécu pendant tout le projet sans conséquence
visible. C'est précisément ce qui la rendait dangereuse : elle n'aurait fait de dégâts que chez
la personne qui aurait tenté de reprendre le projet.

## Prochaine étape

Phase 9, présentation et soutenance.